# Topic 2 — RAG SERP Analyzer (Skill + RAG + Proposal Generator)

Self-contained Colab port of `topic-2-rag-serp-analyzer/` from the
`project-2026-01-claude` repo, as of 2026-09-12.

**Architecture** (unchanged from the source project — see
`docs/component-specs.md`):
- **Deterministic control flow** — `src/orchestrator.py`, `src/skill.py`
  (mostly), `src/manual_store.py`, `src/schemas.py`. Ported **verbatim**
  from disk, not retyped — every code cell below is read directly from the
  real source files.
- **Dynamic components** — two LLM calls, both backed by Azure OpenAI:
  content-gap detection (part of the "SERP Analyzer Skill," which is
  **hybrid** — heading extraction and keyword counting are deterministic,
  gap detection is not) and the Proposal Generator (combines the SERP
  analysis with retrieved manual guidance into the final proposal).
- **Embeddings** — also Azure OpenAI, but a *separate* deployment
  (`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`) from the chat model, and a
  genuinely different API shape (text-in/vector-out, no persona/judgment) —
  see `docs/component-specs.md`'s classification table for why this isn't
  counted as "dynamic" the way the chat calls are.
- **Vector store** — ChromaDB, running locally inside this Colab runtime
  (ephemeral — re-run the ingestion cell each session).

**Unlike topic 1, there's no human-in-the-loop checkpoint** — this topic's
sourced requirements have no equivalent of topic 1's SEL step, so the whole
pipeline runs straight through from keyword to proposal.

**Before running:** open the 🔑 Secrets panel in Colab's left sidebar and add
five secrets, then grant this notebook access to each: `AZURE_OPENAI_API_KEY`,
`AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_VERSION`, `AZURE_OPENAI_DEPLOYMENT`
(chat), and `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` (embeddings — genuinely used
here, unlike topic 1).

In [13]:
!pip install -q "pydantic>=2.0.0" "openai>=1.40.0" "chromadb>=0.5.0"


## 1. Credentials — read from Colab Secrets, never typed or saved to disk

In [14]:
import hashlib
import json
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Protocol

from pydantic import BaseModel, Field, field_validator
from google.colab import userdata

REQUIRED_SECRETS = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT",
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
]

for _name in REQUIRED_SECRETS:
    try:
        os.environ[_name] = userdata.get(_name)
    except userdata.SecretNotFoundError:
        raise RuntimeError(
            f"Colab secret '{_name}' isn't set. Add it via the 🔑 Secrets "
            "panel in the left sidebar, then re-run this cell."
        ) from None
    except userdata.NotebookAccessError:
        raise RuntimeError(
            f"Colab secret '{_name}' exists but this notebook hasn't been "
            "granted access yet — toggle notebook access on for it in the "
            "Secrets panel, then re-run this cell."
        ) from None

print("Loaded Azure OpenAI config from Colab secrets:", ", ".join(REQUIRED_SECRETS))


Loaded Azure OpenAI config from Colab secrets: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_DEPLOYMENT, AZURE_OPENAI_EMBEDDING_DEPLOYMENT


## 2. Data — seeded from the project's real fixtures

`data/SERP_Data.json` and `data/Manual.txt`'s content, embedded verbatim below (Colab has no access to your local `data/` folder), written out to local files here so every function below uses the exact same file-path signatures as the real code — no code is different because it's running in Colab.

In [15]:
Path("data").mkdir(exist_ok=True)
Path("data/SERP_Data.json").write_text('[\n  {\n    "rank": 1,\n    "title": "2026房屋二胎利率總整理：銀行、民間利息與額度試算完整比較",\n    "h2": [\n      "什麼是房屋二胎？",\n      "2026 各大銀行二胎房貸利率比較表",\n      "銀行二胎 vs. 民間二胎：該如何選擇？",\n      "房屋二胎申請流程與必備文件"\n    ],\n    "snippet": "提供全台最新房屋二胎資訊，包含台新、國泰等銀行二胎利率試算。教你如何在 10 分鐘內判斷自己的條件適合哪種貸款方案...",\n    "source_authority": "High (Financial Portal)"\n  },\n  {\n    "rank": 2,\n    "title": "二胎房貸風險大公開！資深代書揭露 5 個常見的合法陷阱",\n    "h2": [\n      "二胎房貸真的安全嗎？",\n      "小心這 3 種高利貸話術",\n      "如何辨識合法的代書事務所",\n      "被騙了怎麼辦？法律救濟途徑說明"\n    ],\n    "snippet": "二胎房貸雖然撥款快，但背後隱藏的法律風險不可不知。本文由執業 20 年代書親自撰寫，教你避開非法融資陷阱...",\n    "source_authority": "Medium (Professional Blog)"\n  },\n  {\n    "rank": 3,\n    "title": "房屋二胎最快 24 小時撥款：適合急用錢、信用瑕疵者的救急方案",\n    "h2": [\n      "急需資金？房屋二胎的 3 大優勢",\n      "免看信用、免入流水：民間二胎的申請門檻",\n      "撥款流程詳解：從估價到拿錢要多久？",\n      "成功案例分享：中小企業主如何透過二胎翻身"\n    ],\n    "snippet": "信用不好也能辦理房屋二胎。提供最高 9 成額度，無須綁約，隨借隨還。針對急用資金需求者提供最快速的專業服務...",\n    "source_authority": "Low (Private Lender)"\n  },\n  {\n    "rank": 4,\n    "title": "銀行二胎房貸申請手冊：低利率、長期數的申辦攻略",\n    "h2": [\n      "為什麼優先選擇銀行二胎？",\n      "影響銀行二胎核貸率的 3 大關鍵",\n      "房屋殘值如何計算？教你自行評估額度",\n      "與一胎房貸轉增貸的差異分析"\n    ],\n    "snippet": "追求穩定與低成本？銀行二胎房貸是首選。詳細解析銀行審核標準，並提供免費線上額度評估工具，讓您資金調度更從容...",\n    "source_authority": "High (Major Bank)"\n  },\n  {\n    "rank": 5,\n    "title": "PTT/Dcard 網友房屋二胎心得：申辦前你必須知道的 10 件事",\n    "h2": [\n      "網友心得：我為什麼後悔辦二胎？",\n      "PTT 常見 Q&A：利率多少算合理？",\n      "代辦公司與直接找金主的差別",\n      "房屋二胎常見失敗原因分析"\n    ],\n    "snippet": "彙整各大社群論壇真實案例，揭露最真實的二胎房貸辦理心得。不要等簽約了才後悔，先看網友遇到的真實痛點...",\n    "source_authority": "Medium (Community Forum)"\n  }\n]', encoding="utf-8")
Path("data/Manual.txt").write_text('關鍵要求：所有關於利率的描述，必須標註「需視個人信用條件而定」。\nEEAT 規範：必須提及「銀行」與「代書/民間」二胎的法律權益差異。\n禁忌：嚴禁出現「保證過件」、「全台最低利」等誇大字眼。', encoding="utf-8")
print("Seeded data/SERP_Data.json and data/Manual.txt")


Seeded data/SERP_Data.json and data/Manual.txt


## 3. Schemas — pydantic contracts for each stage's output

Ported verbatim from `topic-2-rag-serp-analyzer/src/schemas.py`.

In [16]:
class SerpResult(BaseModel):
    """One competitor's SERP entry, exactly as shaped in data/SERP_Data.json.
    Source: B4 (Reference — the fixed input data for this task)."""

    rank: int
    title: str
    h2: List[str] = Field(default_factory=list)
    snippet: str = ""
    source_authority: str = ""


class HeadingExtraction(BaseModel):
    """C2's deterministic heading-structure output, one per SERP result.
    Source: B6."""

    rank: int
    title: str
    h2: List[str]


class KeywordCount(BaseModel):
    """C2's deterministic keyword-distribution output, one per SERP result.
    Source: B7."""

    rank: int
    count: int


class ContentGap(BaseModel):
    """One entry in C2's dynamic gap-detection output. Source: B8."""

    pain_point: str
    evidence: str
    checked_against: List[int] = Field(default_factory=list)


class ContentGapsOutput(BaseModel):
    """Output of C2's dynamic gap-detection call
    (prompts/serp-content-gap-detection.md). Source: B8.

    No minimum-count validator (unlike topic 1's StageAOutput requiring
    >=4 micro-intents) — B8 has no analogous "at least N" requirement;
    zero gaps is a legitimate result, not a validation failure.
    """

    gaps: List[ContentGap] = Field(default_factory=list)


class SerpAnalysisBundle(BaseModel):
    """C2's full output: headings + keyword distribution + gaps, assembled
    by analyze_serp() in skill.py. Source: B5 (+B6, B7, B8)."""

    headings: List[HeadingExtraction]
    keyword_distribution: List[KeywordCount]
    gaps: List[ContentGap]


class RetrievedPassage(BaseModel):
    """One passage returned by C4's retriever. Source: B11."""

    text: str
    score: float
    source_line_range: str


def _coerce_string_list(v: object) -> object:
    """Defensive coercion for recommended_headings/compliance_notes,
    tied to a real failure: despite the prompt asking for plain strings,
    a live Azure run once returned each item as a single-key object instead
    (e.g. {"heading": "..."} or {"note": "..."}) — presumably because the
    prompt's own instructions ask each item to "cite" its source, and the
    model chose to represent that as a structured field rather than folding
    it into the string's own text. The prompt (proposal-generation.md) was
    tightened to say explicitly not to do this — this validator is the B21
    (Technical Rigor) safety net for whenever a model still does anyway,
    not a substitute for the prompt fix. Only unwraps the specific shape
    actually observed (a dict); a genuinely different malformed shape
    (not a str, not a dict) is left as-is for pydantic's normal
    string_type ValidationError to catch and report clearly.
    """
    if not isinstance(v, list):
        return v
    out = []
    for item in v:
        if isinstance(item, dict):
            out.append("; ".join(str(val) for val in item.values()) if item else "")
        else:
            out.append(item)
    return out


def _coerce_string(v: object) -> object:
    """Same defensive coercion as _coerce_string_list, for keyword_guidance
    specifically (a single string, not a list) — a live run once returned
    this as a multi-key object (e.g. {"primary_keyword": ..., ...}) rather
    than the one required string. Joins every value into one readable
    string rather than guessing which key was "the" answer."""
    if isinstance(v, dict):
        return "; ".join(f"{k}: {val}" for k, val in v.items())
    return v


class ProposalOutput(BaseModel):
    """Output of C5, the Proposal Generator
    (prompts/proposal-generation.md). Source: B13 (+B14, B22).

    `manual_silent` is the schema-level home for the self-check both
    prompts.py's proposal_prompt() and the prompt template itself require:
    if C4 returned no relevant passages, this must be True rather than the
    model quietly inventing compliance guidance it wasn't actually given.
    """

    recommended_headings: List[str]
    keyword_guidance: str
    compliance_notes: List[str] = Field(default_factory=list)
    manual_silent: bool = False

    @field_validator("recommended_headings", "compliance_notes", mode="before")
    @classmethod
    def _coerce_headings_and_notes(cls, v: object) -> object:
        return _coerce_string_list(v)

    @field_validator("keyword_guidance", mode="before")
    @classmethod
    def _coerce_keyword_guidance(cls, v: object) -> object:
        return _coerce_string(v)


## 4. Prompt templates + prompt-rendering functions

The two templates below are embedded **verbatim** from `topic-2-rag-serp-analyzer/prompts/*.md`. The rendering functions are ported verbatim from `src/prompts.py`.

In [17]:
GAP_DETECTION_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=2, component C2b
Source: topic-2-rag-serp-analyzer/docs/problem-statement-categorized.md
  Goal: B8 (content-gap detection half of the SERP Analyzer Skill, B5)
Type: dynamic (wrapped by C2's deterministic shell — see component-specs.md)

Run note: B8 ("content gaps — user pain points none of the top-5 address")
is not a deterministic field-match against data/SERP_Data.json (checked its
actual shape: title/h2/snippet/source_authority, no pain-point vocabulary to
match against). Identifying an implicit, unaddressed reader need from
unstructured heading/snippet text requires semantic judgment, the same
category of task as topic 1's Stage A micro-intent inference — so this one
piece of the Skill is dynamic, called from within an otherwise-deterministic
function (`analyze_serp`), not a standalone top-level component.
-->

# SERP Analyzer — Content Gap Detection

## Role
You are a senior SEO systems engineer: someone who combines deep, practical SEO
knowledge with the ability to design and build the AI systems that act on that
knowledge. For this system, you understand that competitive SERP signal and
internal compliance guidance carry different authority — one tells you what
ranks, the other tells you what's allowed to be published — and you build
retrieval and prompting logic that keeps both visible rather than letting
either one silently dominate.

## Goal
Given the top-ranking competitors' extracted heading structure and snippets
for a keyword, identify genuine user pain points that none of them address. (B8)

## Instructions
- **Constraints:**
  - Only surface a gap if it is actually absent across *all* of the given
    results — not just under-emphasized by one (B8's "none of the top-5"
    wording is exact, not approximate).
  - Every gap must cite the specific evidence for why it's a real, checkable
    absence — which headings/snippets were reviewed and what a covering
    result would have needed to say — not a vague "this could be more
    thorough" (ties to B21 — Technical Rigor extends to this dynamic step
    too, not just the deterministic parsing around it).
  - Do not invent a pain point unrelated to the actual keyword/domain context
    given; every gap must be something a real searcher for this keyword would
    plausibly have.
- **Reference:**
  - Input: this stage receives the deterministically-extracted heading
    structure (title + h2 per result) and snippet text for the top-ranking
    results — not raw HTML, and not the full page content (component C2's
    deterministic shell has already done that extraction; see
    `component-specs.md`).
  - The keyword this analysis is being run for.
- **Output shape** (feeds into C1's orchestrator, then C5's Proposal
  Generator — see `component-specs.md`):
  - `{"gaps": [{"pain_point": ..., "evidence": ..., "checked_against": [list of ranks reviewed]}, ...]}`.
  - No minimum or maximum count required (unlike topic 1's ≥4 micro-intents) —
    report however many genuine gaps are actually found, including zero if
    the top results already cover the space well.
- **Self-check before finalizing** (from B21):
  - For every gap, could a reader trace it back to specific absent content in
    the given headings/snippets, or is it a plausible-sounding guess?
  - Would this same gap disappear if you'd only looked at 4 of the 5 results
    instead of all 5? If so, it isn't actually a gap across the full set."""

PROPOSAL_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=2, component C5
Source: topic-2-rag-serp-analyzer/docs/problem-statement-categorized.md
  Goal: B13 (LLM+RAG+Skill integration) driving B14 (the generated proposal
  the frontend displays)
Type: dynamic
-->

# SEO Article Planning Proposal Generator

## Role
You are a senior SEO systems engineer: someone who combines deep, practical SEO
knowledge with the ability to design and build the AI systems that act on that
knowledge. For this system, you understand that competitive SERP signal and
internal compliance guidance carry different authority — one tells you what
ranks, the other tells you what's allowed to be published — and you build
retrieval and prompting logic that keeps both visible rather than letting
either one silently dominate.

## Goal
For a given keyword, produce an SEO article planning proposal (SEO 文章規劃建議書)
that an editor could act on directly — combining the SERP Analyzer Skill's
competitive analysis with the internal writing manual's retrieved compliance
and style guidance. (B13, driving B14)

## Instructions
- **Constraints:**
  - Must actually combine both sources, not lean on one and mention the other
    in passing (B22 — Prompt Precision is graded on this directly). Every
    heading recommendation should trace to either a content gap or a
    competitive pattern from the SERP analysis; every compliance note should
    trace to an actual retrieved manual passage, not a general SEO best
    practice you already knew.
  - If a compliance rule from the manual would prohibit a competitive tactic
    the SERP data suggests (e.g. a headline pattern that uses guarantee-style
    language), the compliance rule wins — say so explicitly rather than
    quietly omitting the tactic.
  - Do not fabricate a compliance justification when the retrieved passages
    don't actually cover this keyword's specific angle — say the manual is
    silent on that point rather than inventing plausible-sounding guidance.
- **Reference:**
  - Input: the keyword; C2's SERP analysis bundle (heading structure, keyword
    distribution, detected content gaps); C4's retrieved manual passages
    (with their source line ranges) for this keyword.
  - B2 — the system must weigh SERP competitive signal and internal
    compliance guidance together, not either in isolation; this is the
    component where that actually happens.
- **Output shape** (feeds B14's frontend display, and B18's architecture
  diagram's final data-flow node) — every field below is a **plain string
  or a list of plain strings, never a nested object**. If a heading or
  compliance note needs to cite its source (a content gap, a competitor
  pattern, a manual passage), write that citation as part of the string's
  own text — do not represent it as a separate key on an object:
  - `recommended_headings`: a list of plain heading strings (informed by
    the content gaps and competitor heading patterns) — e.g.
    `"提前清償與轉貸限制常見問題（因應競品未涵蓋的內容缺口）"`, not
    `{"heading": "...", "source": "gap"}`.
  - `keyword_guidance`: **one single string**, not an object with its own
    sub-fields — fold every point (primary keyword, placement, density
    caveats) into that one string's text.
  - `compliance_notes`: a list of plain strings, each one citing which
    retrieved manual passage it's grounded in as part of the string's own
    text (e.g. `"...（Manual 第1行）"`), not `{"note": "...", "source": "..."}`.
  - `manual_silent`: an explicit `true`/`false` — true when the manual
    returned no relevant guidance for this keyword, rather than silently
    omitting the compliance section.
- **Self-check before finalizing** (from B22):
  - If you removed the SERP analysis entirely, would this proposal change?
    If not, you're not actually using it.
  - If you removed the retrieved manual passages entirely, would this
    proposal change? If not, you're not actually using them either.
  - Does every compliance claim point at a specific retrieved passage, or
    are any of them just general SEO knowledge dressed up as manual guidance?"""


def gap_detection_prompt(keyword: str, serp_context: list[dict[str, Any]]) -> str:
    """Renders prompts/serp-content-gap-detection.md. `serp_context` is the
    deterministically-extracted {rank, title, h2, snippet} per result — not
    raw HTML, per that template's Reference section."""
    template = GAP_DETECTION_TEMPLATE
    return (
        f"{template}\n\n---\nKeyword: {keyword}\n\n"
        "Extracted heading structure and snippets for the top-ranking results:\n"
        f"{json.dumps(serp_context, ensure_ascii=False, indent=2)}\n\n"
        'Respond with ONLY a JSON object: {"gaps": [{"pain_point": ..., '
        '"evidence": ..., "checked_against": [...]}, ...]}.'
    )


def proposal_prompt(
    keyword: str,
    serp_analysis: dict[str, Any],
    retrieved_passages: list[dict[str, Any]],
) -> str:
    """Renders prompts/proposal-generation.md. `retrieved_passages` may be
    an empty list — that's a legitimate input meaning C4 found nothing
    relevant, which the template's Constraints require surfacing as
    `manual_silent: true`, not silently omitting."""
    template = PROPOSAL_TEMPLATE
    return (
        f"{template}\n\n---\nKeyword: {keyword}\n\n"
        "SERP analysis bundle (headings, keyword distribution, content gaps):\n"
        f"{json.dumps(serp_analysis, ensure_ascii=False, indent=2)}\n\n"
        "Retrieved manual passages (empty list means the manual had nothing relevant):\n"
        f"{json.dumps(retrieved_passages, ensure_ascii=False, indent=2)}\n\n"
        'Respond with ONLY a JSON object of this EXACT shape — every list item '
        'a plain string, never an object, and keyword_guidance a single plain '
        'string, never an object with its own sub-fields: '
        '{"recommended_headings": ["heading text with any citation folded in", '
        '"..."], "keyword_guidance": "one string covering all keyword-usage '
        'points", "compliance_notes": ["note text with its manual citation '
        'folded in", "..."], "manual_silent": bool}.'
    )


## 5. JSON parsing gate

Ported verbatim from `src/json_utils.py`.

In [18]:
def parse_json_response(raw: str) -> Any:
    """Each dynamic component is instructed to return JSON; this is where
    that contract is enforced rather than trusted blindly (ties to B21 —
    Technical Rigor)."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Component did not return valid JSON: {exc}\nRaw: {raw!r}") from exc


## 6. LLM + Embedding clients — Azure OpenAI backend

Ported **verbatim** from `src/llm_client.py` — no adaptation needed this time (unlike topic 1's notebook): both classes already read config from `os.environ`, which Cell 1 above already populated from Colab Secrets.

In [19]:
class LLMClient(Protocol):
    def complete(self, prompt: str, *, component: str) -> str: ...


class EmbeddingClient(Protocol):
    def embed(self, texts: List[str]) -> List[List[float]]: ...


def _require_env(names: list[str]) -> Dict[str, str]:
    """Shared validation for both Azure clients below: every named var must
    be present and not an obvious .env.example placeholder ("your-...", or
    containing "<" as in "<your-resource>"). Raises once, naming every
    missing/placeholder var, rather than failing on the first one and
    forcing a fix-one-rerun-fix-the-next cycle."""
    values: Dict[str, str] = {}
    missing = []
    for name in names:
        val = os.environ.get(name, "")
        if not val or val.startswith("your-") or "<" in val:
            missing.append(name)
        else:
            values[name] = val
    if missing:
        raise RuntimeError(
            "Missing real .env values for: " + ", ".join(missing)
        )
    return values


class AzureOpenAILLMClient:
    """Chat-completion backend for the two dynamic components. Requests
    Azure/OpenAI JSON mode for both — unlike topic 1, *every* dynamic
    component here returns structured JSON (neither one is free-text like
    topic 1's Stage B), so there's no free-text exception list."""

    def __init__(self) -> None:
        vals = _require_env([
            "AZURE_OPENAI_API_KEY",
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_DEPLOYMENT",
        ])
        self._key = vals["AZURE_OPENAI_API_KEY"]
        self._endpoint = vals["AZURE_OPENAI_ENDPOINT"]
        self._api_version = vals["AZURE_OPENAI_API_VERSION"]
        self._deployment = vals["AZURE_OPENAI_DEPLOYMENT"]

    def complete(self, prompt: str, *, component: str) -> str:
        from openai import AzureOpenAI  # imported lazily so this module loads without the package installed

        client = AzureOpenAI(
            api_key=self._key,
            azure_endpoint=self._endpoint,
            api_version=self._api_version,
        )
        response = client.chat.completions.create(
            model=self._deployment,  # Azure takes the deployment name here, not a model name
            messages=[{"role": "user", "content": prompt}],
            # max_completion_tokens, not max_tokens — see topic 1's
            # llm_client.py for why (newer/GPT-5-class deployments reject
            # max_tokens outright).
            max_completion_tokens=2048,
            response_format={"type": "json_object"},
        )
        return response.choices[0].message.content


class AzureOpenAIEmbeddingClient:
    """Embedding backend for manual ingestion (C3) and retrieval (C4). Uses
    AZURE_OPENAI_EMBEDDING_DEPLOYMENT — a separate deployment from the chat
    model above; Azure serves embeddings and chat completions from
    different deployed models, so this can't reuse AZURE_OPENAI_DEPLOYMENT."""

    def __init__(self) -> None:
        vals = _require_env([
            "AZURE_OPENAI_API_KEY",
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
        ])
        self._key = vals["AZURE_OPENAI_API_KEY"]
        self._endpoint = vals["AZURE_OPENAI_ENDPOINT"]
        self._api_version = vals["AZURE_OPENAI_API_VERSION"]
        self._deployment = vals["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

    def embed(self, texts: List[str]) -> List[List[float]]:
        from openai import AzureOpenAI  # imported lazily, same reason as above

        client = AzureOpenAI(
            api_key=self._key,
            azure_endpoint=self._endpoint,
            api_version=self._api_version,
        )
        response = client.embeddings.create(model=self._deployment, input=texts)
        return [item.embedding for item in response.data]


class FixtureLLMClient:
    """Same pattern as topic 1's: queued pre-recorded responses per
    component, consumed in order — so repeated calls to the same component
    each get their own recorded response."""

    def __init__(self, fixtures: Dict[str, list]) -> None:
        self._fixtures = {k: list(v) for k, v in fixtures.items()}
        self.calls: list[str] = []

    def complete(self, prompt: str, *, component: str) -> str:
        self.calls.append(component)
        queue = self._fixtures.get(component)
        if not queue:
            raise KeyError(f"No fixture left for component '{component}'")
        return queue.pop(0)


class FixtureEmbeddingClient:
    """Deterministic fake embeddings — no real API call, no network. Hashes
    each text into a fixed-size float vector so identical text always
    yields an identical vector and different text yields a different one,
    which is enough to exercise Chroma's storage/similarity-search logic in
    tests without a live embedding endpoint. Not a semantically meaningful
    embedding — never use for anything but control-flow testing."""

    DIM = 16

    def embed(self, texts: List[str]) -> List[List[float]]:
        vectors = []
        for t in texts:
            digest = hashlib.sha256(t.encode("utf-8")).digest()
            vectors.append([b / 255.0 for b in digest[: self.DIM]])
        return vectors


## 7. C2 — SERP Analyzer Skill (hybrid)

Ported verbatim from `src/skill.py`. Heading extraction and keyword counting are deterministic; `detect_content_gaps` wraps one dynamic call.

In [20]:
def extract_headings(serp_data: list[SerpResult]) -> list[HeadingExtraction]:
    """Deterministic. Source: B6. A malformed individual result (missing
    h2/title) would fail at SerpResult validation before reaching here —
    see orchestrator.load_serp_data — so this function itself can assume
    well-formed input per result."""
    return [HeadingExtraction(rank=r.rank, title=r.title, h2=r.h2) for r in serp_data]


def keyword_distribution(serp_data: list[SerpResult], keyword: str) -> list[KeywordCount]:
    """Deterministic. Source: B7. Case-insensitive substring count of
    `keyword` across each result's title + h2 + snippet combined — a
    simple, auditable heuristic; not stemming/tokenization-aware, which is
    an acceptable simplification for a prototype but worth knowing if
    keyword variants (e.g. plurals, particles) need to count too."""
    out = []
    kw = keyword.lower()
    for r in serp_data:
        haystack = " ".join([r.title, " ".join(r.h2), r.snippet]).lower()
        out.append(KeywordCount(rank=r.rank, count=haystack.count(kw)))
    return out


def detect_content_gaps(serp_data: list[SerpResult], keyword: str, llm: LLMClient) -> list[ContentGap]:
    """Hybrid. Source: B8. Deterministically assembles the dynamic call's
    input (heading structure + snippets, not raw HTML — see
    prompts/serp-content-gap-detection.md's Reference section), then
    validates the returned shape before returning it."""
    context = [
        {"rank": r.rank, "title": r.title, "h2": r.h2, "snippet": r.snippet}
        for r in serp_data
    ]
    raw = llm.complete(gap_detection_prompt(keyword, context), component="gap_detection")
    validated = ContentGapsOutput.model_validate(parse_json_response(raw))
    return validated.gaps


def analyze_serp(serp_data: list[SerpResult], keyword: str, llm: LLMClient) -> SerpAnalysisBundle:
    """C2's full entry point, called by C1. Source: B5."""
    return SerpAnalysisBundle(
        headings=extract_headings(serp_data),
        keyword_distribution=keyword_distribution(serp_data, keyword),
        gaps=detect_content_gaps(serp_data, keyword, llm),
    )


## 8. C3/C4 — Manual Ingestion + Retriever

Ported verbatim from `src/manual_store.py`, including the per-line chunking default (chosen after checking the real `Manual.txt` — see `docs/component-specs.md`'s run notes) and the explicit cosine distance metric.

In [21]:
COLLECTION_NAME = "topic2_manual"


def _chunk_manual(text: str) -> list[dict[str, str]]:
    """One chunk per non-empty line. Engineering default (see
    docs/component-specs.md run notes), not a sourced requirement —
    originally planned as paragraph-level (split on blank lines), but the
    actual data/Manual.txt has three distinct compliance rules with *no*
    blank lines between them, which would degenerate paragraph-splitting
    to a single chunk covering the whole file — losing per-rule provenance
    entirely (checked the real file before picking this, not assumed).
    Per-line chunking gives each rule its own retrievable, citable chunk for
    this manual's actual shape. `[needs review]` if a future, longer manual
    has rules that wrap across multiple lines — this would then fragment a
    single rule mid-sentence, and a real paragraph- or sentence-aware
    splitter would be needed instead."""
    chunks: list[dict[str, str]] = []
    for i, line in enumerate(text.splitlines(), start=1):
        if line.strip():
            chunks.append({"text": line.strip(), "source_line_range": str(i)})
    return chunks


def get_chroma_client(persist_dir: str = ".chroma"):
    """Lazily imported so this module loads without chromadb installed."""
    import chromadb

    return chromadb.PersistentClient(path=persist_dir)


def ingest_manual(manual_path: str, embedding_client: EmbeddingClient, persist_dir: str = ".chroma") -> None:
    """C3. Source: B9 (+B10). Runs offline/once (or whenever Manual.txt
    changes) — not part of the per-request path; see
    docs/component-specs.md's orchestration diagram.

    Drops and recreates the collection on every call rather than appending,
    so re-running after Manual.txt changes doesn't silently accumulate
    stale chunks alongside the new ones.
    """
    # utf-8-sig, not utf-8 — data/Manual.txt carries a UTF-8 BOM (same as
    # data/SERP_Data.json), which would otherwise land as a stray invisible
    # character prefixing the first chunk.
    text = Path(manual_path).read_text(encoding="utf-8-sig").strip()
    if not text:
        raise ValueError(
            f"{manual_path} is empty — refusing to silently populate an "
            "empty collection that C4 would then query against forever "
            "without anyone noticing (B21)."
        )

    chunks = _chunk_manual(text)
    client = get_chroma_client(persist_dir)
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass  # collection didn't exist yet — fine, create() below handles it
    # Explicit cosine distance — Chroma's default (squared L2) makes the
    # "score" approximation below uninterpretable (can go negative); cosine
    # is also the standard metric for the kind of normalized text
    # embeddings Azure OpenAI's embedding models produce.
    collection = client.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

    vectors = embedding_client.embed([c["text"] for c in chunks])
    collection.add(
        ids=[str(i) for i in range(len(chunks))],
        embeddings=vectors,
        documents=[c["text"] for c in chunks],
        metadatas=[{"source_line_range": c["source_line_range"]} for c in chunks],
    )


def retrieve_manual_guidance(
    keyword: str,
    embedding_client: EmbeddingClient,
    top_k: int = 3,
    persist_dir: str = ".chroma",
) -> List[RetrievedPassage]:
    """C4. Source: B11. `top_k=3` is an engineering default (see
    docs/component-specs.md run notes), not a sourced requirement.

    Raises if the collection doesn't exist (ingest_manual() never ran) —
    this must be distinguishable from "genuinely no relevant guidance for
    this keyword" (a valid, informative empty-list result), not conflated
    with it.
    """
    client = get_chroma_client(persist_dir)
    try:
        collection = client.get_collection(COLLECTION_NAME)
    except Exception as exc:
        raise RuntimeError(
            "Manual collection not found — call ingest_manual() first "
            "(it hasn't run yet, or persist_dir doesn't match)."
        ) from exc

    [query_vector] = embedding_client.embed([keyword])
    results = collection.query(query_embeddings=[query_vector], n_results=top_k)

    passages: List[RetrievedPassage] = []
    docs = results["documents"][0] if results["documents"] else []
    metas = results["metadatas"][0] if results["metadatas"] else []
    dists = results["distances"][0] if results["distances"] else []
    for doc, meta, dist in zip(docs, metas, dists):
        # Chroma's default distance is squared L2 by our (unspecified) index
        # config; treating (1 - dist) as a similarity "score" is a rough
        # approximation good enough for a prototype's provenance display,
        # not a calibrated confidence value.
        passages.append(
            RetrievedPassage(text=doc, score=1 - dist, source_line_range=meta["source_line_range"])
        )
    return passages


## 9. C1 — Orchestrator

Ported verbatim from `src/orchestrator.py`.

In [22]:
@dataclass
class ProposalResult:
    keyword: str
    serp_analysis: dict[str, Any]
    retrieved_passages: list[dict[str, Any]] = field(default_factory=list)
    proposal: dict[str, Any] = field(default_factory=dict)


def load_serp_data(path: str) -> list[SerpResult]:
    """Source: B4. Raises a clear, wrapped error on a missing/malformed
    file rather than letting a bare FileNotFoundError/JSONDecodeError
    surface — matches component-specs.md's C1 error-handling requirement
    (B21): don't silently proceed with an empty competitive analysis.
    Uses utf-8-sig since data/SERP_Data.json carries a UTF-8 BOM."""
    try:
        raw = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    except (FileNotFoundError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"Could not load SERP data from {path}: {exc}") from exc
    return [SerpResult.model_validate(r) for r in raw]


def generate_seo_proposal(
    keyword: str,
    serp_data_path: str,
    llm: LLMClient,
    embedding_client: EmbeddingClient,
    top_k: int = 3,
    persist_dir: str = ".chroma",
) -> ProposalResult:
    """C1's entry point. Source: B3 (umbrella), B13.

    Logic (see docs/component-specs.md's orchestration diagram):
    1. Load SERP data (deterministic, fails loudly on bad input).
    2. Call C2 analyze_serp() — headings + keyword distribution
       (deterministic) + content gaps (dynamic, validated).
    3. Call C4 retrieve_manual_guidance() — vector search against C3's
       pre-populated collection; may legitimately return [].
    4. Call C5 (dynamic) with both bundles together — an empty passages
       list is passed through as-is, not special-cased here: the prompt's
       own Constraints require it to say the manual is silent rather than
       inventing guidance, so the orchestrator doesn't need a second gate
       for that.
    """
    serp_data = load_serp_data(serp_data_path)
    analysis = analyze_serp(serp_data, keyword, llm)
    passages = retrieve_manual_guidance(keyword, embedding_client, top_k=top_k, persist_dir=persist_dir)

    raw_proposal = llm.complete(
        proposal_prompt(
            keyword,
            analysis.model_dump(),
            [p.model_dump() for p in passages],
        ),
        component="proposal_generation",
    )
    proposal = ProposalOutput.model_validate(parse_json_response(raw_proposal))

    return ProposalResult(
        keyword=keyword,
        serp_analysis=analysis.model_dump(),
        retrieved_passages=[p.model_dump() for p in passages],
        proposal=proposal.model_dump(),
    )


## 10. Ingest the manual (run once per session)

This is the offline/setup step from `docs/component-specs.md`'s orchestration diagram — not part of the per-keyword request path.

In [23]:
embed_client = AzureOpenAIEmbeddingClient()
ingest_manual("data/Manual.txt", embed_client, persist_dir=".chroma")
print("Manual ingested into ./.chroma")


Manual ingested into ./.chroma


## 11. Run it — generate a proposal for a keyword

Change `KEYWORD` below to try a different one. This makes real Azure OpenAI calls: one for gap detection, one for proposal generation, plus the embedding call already made above during ingestion and one more here to embed the keyword for retrieval.

In [24]:
KEYWORD = "房屋二胎利率"  # change this to try a different keyword

llm_client = AzureOpenAILLMClient()
result = generate_seo_proposal(
    KEYWORD, "data/SERP_Data.json", llm_client, embed_client, top_k=3, persist_dir=".chroma"
)

print("=== SERP analysis ===")
print(json.dumps(result.serp_analysis, ensure_ascii=False, indent=2))
print("\n=== Retrieved manual passages ===")
print(json.dumps(result.retrieved_passages, ensure_ascii=False, indent=2))
print("\n=== Proposal ===")
print(json.dumps(result.proposal, ensure_ascii=False, indent=2))


=== SERP analysis ===
{
  "headings": [
    {
      "rank": 1,
      "title": "2026房屋二胎利率總整理：銀行、民間利息與額度試算完整比較",
      "h2": [
        "什麼是房屋二胎？",
        "2026 各大銀行二胎房貸利率比較表",
        "銀行二胎 vs. 民間二胎：該如何選擇？",
        "房屋二胎申請流程與必備文件"
      ]
    },
    {
      "rank": 2,
      "title": "二胎房貸風險大公開！資深代書揭露 5 個常見的合法陷阱",
      "h2": [
        "二胎房貸真的安全嗎？",
        "小心這 3 種高利貸話術",
        "如何辨識合法的代書事務所",
        "被騙了怎麼辦？法律救濟途徑說明"
      ]
    },
    {
      "rank": 3,
      "title": "房屋二胎最快 24 小時撥款：適合急用錢、信用瑕疵者的救急方案",
      "h2": [
        "急需資金？房屋二胎的 3 大優勢",
        "免看信用、免入流水：民間二胎的申請門檻",
        "撥款流程詳解：從估價到拿錢要多久？",
        "成功案例分享：中小企業主如何透過二胎翻身"
      ]
    },
    {
      "rank": 4,
      "title": "銀行二胎房貸申請手冊：低利率、長期數的申辦攻略",
      "h2": [
        "為什麼優先選擇銀行二胎？",
        "影響銀行二胎核貸率的 3 大關鍵",
        "房屋殘值如何計算？教你自行評估額度",
        "與一胎房貸轉增貸的差異分析"
      ]
    },
    {
      "rank": 5,
      "title": "PTT/Dcard 網友房屋二胎心得：申辦前你必須知道的 10 件事",
      "h2": [
        "網友心得：我為什麼後悔辦二胎？",
        "PTT 常見 Q&A：利率